In [1]:
using DelimitedFiles
include("open_optimization_problem.jl")   # pulls in the full include chain

n      = 3
J      = fill(1/4, n - 1)
gammas = fill(J[1]/4, n)
tlist  = range(0, 25; length = 100)
excited = ["0"]
ks     = [2, 8, 15]
dissipation = false

cutoff, maxdim = 0.0, 16     # was 1e-10, 200 — maxdim=16 is already exact for n=3
order     = 1     # order of the product formulas being combined
k_ref     = 100   # fine reference standing in for e^{tL} in F_ex
order_ref = 4                # was 2

lsites = liouville_siteinds(n)
rho0   = vectorized_initial_state_mps(lsites, excited)

coeffs = zeros(Float64, length(tlist), length(ks))

for (i, t) in enumerate(tlist)
    if t <= 0
        coeffs[i, :] .= NaN
        continue
    end
    M, _ = open_gram_matrix(n, J, gammas, t, ks, lsites, rho0;
                            cutoff = cutoff, maxdim = maxdim,
                            order = order, dissipation = dissipation)
    L, _ = open_L_vector(n, J, gammas, t, ks, k_ref, lsites, rho0;
                         cutoff = cutoff, maxdim = maxdim,
                         order = order, order_ref = order_ref,
                         dissipation = dissipation)
    c, _ = dynamic_mpf_coefficients(M, L)
    coeffs[i, :] .= c
    println("t = ", round(t, digits = 4), "  c = ", c,
            "  sum = ", sum(c), "  cond(M) = ", cond(M))
end

open("n_3_mpf_coefficients_3815.txt", "w") do io
    println(io, "# t\tc_k3\tc_k8\tc_k12")
    writedlm(io, hcat(collect(tlist), coeffs))
end

t = 0.2525  c = [0.0511559196453703, -1.5227966876615233, 2.471640768016153]  sum = 1.0000000000000002  cond(M) = 1.4280389140938477e11
t = 0.5051  c = [0.05140613071208603, -1.524426532336253, 2.473020401624167]  sum = 1.0  cond(M) = 2.305461679629749e9
t = 0.7576  c = [0.05156138059876602, -1.5251958157690102, 2.473634435170244]  sum = 0.9999999999999998  cond(M) = 2.128850683076513e8
t = 1.0101  c = [0.051776156301885, -1.5262476265614282, 2.474471470259543]  sum = 1.0  cond(M) = 4.056696433301666e7
t = 1.2626  c = [0.05204713827606387, -1.5275506541014612, 2.4755035158253973]  sum = 1.0  cond(M) = 1.156185804576931e7
t = 1.5152  c = [0.05236808734169429, -1.5290493506296152, 2.476681263287921]  sum = 1.0  cond(M) = 4.261167167203725e6
t = 1.7677  c = [0.05272816620207421, -1.5306513735918046, 2.4779232073897304]  sum = 1.0  cond(M) = 1.8751369031916976e6
t = 2.0202  c = [0.05311028270803808, -1.5322155773504265, 2.4791052946423884]  sum = 1.0  cond(M) = 937450.2489494316
t = 2.2727